In [1]:
import sys
!{sys.executable} -m pip install flask flask-cors
print("安装完成！")

安装完成！



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import sys
import joblib
import pandas as pd
import numpy as np
import re
import jieba
from flask import Flask, request, jsonify
from collections import Counter
from flask_cors import CORS  # 解决跨域问题


In [3]:
# ============== 1. 加载模型和向量器 ==============
MODEL_DIR = r"E:\\graduation project\\E-commerce Review Sentiment Classification\\models"

print("正在加载模型...")
try:
    model = joblib.load(os.path.join(MODEL_DIR, "best_svm_model.pkl"))
    vectorizer = joblib.load(os.path.join(MODEL_DIR, "tfidf_vectorizer.pkl"))
    print(f"模型加载成功: SVM (F1: 0.9133)")
except Exception as e:
    print(f"模型加载失败: {e}")
    sys.exit(1)

正在加载模型...
模型加载成功: SVM (F1: 0.9133)


In [4]:
# ============== 2. 加载停用词 ==============
STOPWORDS = set()
stopword_file = r"E:\\graduation project\\E-commerce Review Sentiment Classification\\data\\stopwords.txt"
try:
    with open(stopword_file, "r", encoding="utf-8") as f:
        for line in f:
            word = line.strip()
            if word:
                STOPWORDS.add(word)
    print(f"停用词加载成功: {len(STOPWORDS)}个")
except:
    print("停用词加载失败，使用默认停用词")
    STOPWORDS = {"的", "了", "是", "在", "我", "有", "和", "就", "不", "也", "这", "你"}

停用词加载成功: 63个


In [5]:
# ============== 定义预处理函数==============
def clean_text(text):
    """文本清洗"""
    if pd.isna(text):
        return ''
    text = str(text)
    # 去除各类括号
    text = re.sub(r'[【】\[\]（）()《》<>「」『』【】〔〕]', '', text)
    # 去除英文和数字
    text = re.sub(r'[a-zA-Z0-9]', '', text)
    # 去除特殊符号
    text = re.sub(r'[▼◆▶■●★☆◆◇○◎●△▲※→←↑↓]', '', text)
    # 去除空白字符
    text = re.sub(r'\s+', '', text)
    return text

def cut_words(text):
    """分词 + 去停用词 + 过滤短词"""
    words = jieba.lcut(text)
    words = [w for w in words if w not in STOPWORDS and len(w) >= 2]
    return ' '.join(words)

def preprocess_single(text):
    """单条评论预处理完整流程"""
    cleaned = clean_text(text)
    seg_text = cut_words(cleaned)
    return seg_text

def preprocess_for_wordcloud(text):
    """
    preprocess_single 返回的是空格连接的字符串，这里改为返回列表
    """
    if pd.isna(text) or not isinstance(text, str):
        return []
    
    # 复用清洗函数
    cleaned = clean_text(text)
    
    # 复用分词和过滤逻辑，但返回列表
    words = jieba.lcut(cleaned)
    words = [w for w in words if w not in STOPWORDS and len(w) >= 2]
    
    return words

In [6]:
# ============== 4. 创建Flask应用 ==============
app = Flask(__name__)
CORS(app)  # 允许跨域请求

@app.route('/health', methods=['GET'])
def health():
    """健康检查接口"""
    return jsonify({
        'status': 'ok',
        'model': 'SVM',
        'message': '模型服务正常运行'
    })

@app.route('/predict', methods=['POST'])
def predict():
    """单条评论预测接口"""
    try:
        # 获取请求数据
        data = request.get_json()
        if not data or 'text' not in data:
            return jsonify({'error': '请提供text参数'}), 400
        
        text = data['text']
        
        # 预处理
        seg_text = preprocess_single(text)
        
        # 向量化
        vec = vectorizer.transform([seg_text])
        
        # 预测
        pred = model.predict(vec)[0]
        
        # 置信度（SVM需要用decision_function）
        if hasattr(model, 'decision_function'):
            confidence = model.decision_function(vec)[0]
            # 归一化到0-1之间
            confidence = 1 / (1 + np.exp(-confidence))
        else:
            confidence = 0.5
        
        # 返回结果
        return jsonify({
            'code': 200,
            'text': text,
            'sentiment': '好评' if pred == 1 else '差评',
            'label': int(pred),
            'confidence': float(confidence),
            'seg_text': seg_text
        })
        
    except Exception as e:
        return jsonify({'error': str(e)}), 500

@app.route('/batch_predict', methods=['POST'])
def batch_predict():
    """批量评论预测接口"""
    try:
        data = request.get_json()
        if not data or 'texts' not in data:
            return jsonify({'error': '请提供texts参数'}), 400
        
        texts = data['texts']
        if not isinstance(texts, list):
            return jsonify({'error': 'texts必须为列表'}), 400
        
        results = []
        for text in texts:
            seg_text = preprocess_single(text)
            vec = vectorizer.transform([seg_text])
            pred = model.predict(vec)[0]
            
            # 置信度
            if hasattr(model, 'decision_function'):
                confidence = model.decision_function(vec)[0]
                confidence = 1 / (1 + np.exp(-confidence))
            else:
                confidence = 0.5
            
            results.append({
                'text': text,
                'sentiment': '好评' if pred == 1 else '差评',
                'label': int(pred),
                'confidence': float(confidence)
            })
        
        return jsonify({
            'code': 200,
            'total': len(results),
            'results': results
        })
        
    except Exception as e:
        return jsonify({'error': str(e)}), 500


@app.route('/predict/test', methods=['GET'])
def test_predict():
    # 测试预测功能
    print("---预测功能测试---")
    test_cases = [
        "这件衣服质量很好，非常满意！",
        "东西太差了，用了一次就坏了",
        "性价比一般，没有想象中好",
        "发货很快，包装完好",
        "客服态度很差，再也不来了"
    ]
    
    for text in test_cases:
        seg = preprocess_single(text)
        vec = vectorizer.transform([seg])
        pred = model.predict(vec)[0]
        sentiment = '好评' if pred == 1 else '差评'
        print(f"评论: {text}")
        print(f"预测: {sentiment}")
        print(f"分词: {seg}\n")

In [7]:

@app.route('/wordcloud', methods=['POST'])
def generate_wordcloud():
    """
    生成词云数据接口
    请求体格式:
    {
        "positive_texts": ["好评1", "好评2", ...],
        "negative_texts": ["差评1", "差评2", ...],
        "top_n": 50  # 可选，默认50
    }
    返回格式:
    {
        "positive": [{"name": "词1", "value": 10}, ...],
        "negative": [{"name": "词1", "value": 8}, ...]
    }
    """
    try:
        data = request.get_json()
        if not data:
            return jsonify({'error': '请提供数据'}), 400
        
        # 获取参数
        positive_texts = data.get('positive_texts', [])
        negative_texts = data.get('negative_texts', [])
        top_n = data.get('top_n', 50)
        
        # 确保是列表类型
        if not isinstance(positive_texts, list) or not isinstance(negative_texts, list):
            return jsonify({'error': 'positive_texts和negative_texts必须为列表'}), 400
        
        # 处理正面评论
        positive_words = []
        for text in positive_texts:
            positive_words.extend(preprocess_for_wordcloud(text))
        
        # 处理负面评论
        negative_words = []
        for text in negative_texts:
            negative_words.extend(preprocess_for_wordcloud(text))
        
        # 统计词频
        positive_counter = Counter(positive_words)
        negative_counter = Counter(negative_words)
        
        # 转换为前端ECharts词云需要的格式
        positive_result = [
            {"name": word, "value": count}
            for word, count in positive_counter.most_common(top_n)
        ]
        
        negative_result = [
            {"name": word, "value": count}
            for word, count in negative_counter.most_common(top_n)
        ]
        
        return jsonify({
            'code': 200,
            'positive': positive_result,
            'negative': negative_result,
            'total_positive_words': len(positive_words),
            'total_negative_words': len(negative_words)
        })
        
    except Exception as e:
        return jsonify({'error': str(e)}), 500

@app.route('/wordcloud/test', methods=['GET'])
def test_wordcloud():
    print("\n--- 词云功能测试 ---")
    test_positive = [
        "这件衣服质量很好，面料舒适，非常满意",
        "发货很快，包装完好，下次还会购买",
        "性价比很高，比实体店便宜很多",
        "客服态度很好，问题解决很及时",
        "穿上很显瘦，朋友都说好看"
    ]
    
    test_negative = [
        "质量太差了，洗了一次就变形",
        "发货很慢，等了整整一周",
        "客服态度恶劣，再也不来了",
        "尺码偏小，和描述不符",
        "用了一次就坏了，质量堪忧"
    ]
    
    # 处理正面评论
    positive_words = []
    for text in test_positive:
        positive_words.extend(preprocess_single(text))
    
    # 处理负面评论
    negative_words = []
    for text in test_negative:
        negative_words.extend(preprocess_single(text))
    
    positive_counter = Counter(positive_words)
    negative_counter = Counter(negative_words)
    
    return jsonify({
        'positive': [{"name": w, "value": c} for w, c in positive_counter.most_common(20)],
        'negative': [{"name": w, "value": c} for w, c in negative_counter.most_common(20)]
    })

In [8]:
@app.route('/keyword_attribution', methods=['POST'])
def keyword_attribution():
    """
    关键词情感归因接口
    请求体格式:
    {
        "positive_texts": ["好评1", "好评2", ...],
        "negative_texts": ["差评1", "差评2", ...],
        "top_n": 15  # 可选，默认15
    }
    返回格式:
    {
        "positive_keywords": [
            {"keyword": "质量", "count": 10, "examples": ["质量很好", "质量不错"]},
            ...
        ],
        "negative_keywords": [
            {"keyword": "太差", "count": 5, "examples": ["质量太差"]},
            ...
        ]
    }
    """
    try:
        data = request.get_json()
        if not data:
            return jsonify({'error': '请提供数据'}), 400
        
        positive_texts = data.get('positive_texts', [])
        negative_texts = data.get('negative_texts', [])
        top_n = data.get('top_n', 15)
        
        if not isinstance(positive_texts, list) or not isinstance(negative_texts, list):
            return jsonify({'error': 'positive_texts和negative_texts必须为列表'}), 400
        
        def extract_keywords_with_examples(texts):
            """提取关键词并找到包含该关键词的例句"""
            keyword_counter = Counter()
            keyword_examples = {}  # {keyword: [examples]}
            
            for text in texts:
                words = preprocess_for_wordcloud(text)
                keyword_counter.update(words)
                
                for word in words:
                    if word not in keyword_examples:
                        keyword_examples[word] = []
                    if len(keyword_examples[word]) < 3 and text not in keyword_examples[word]:
                        keyword_examples[word].append(text)
            
            result = []
            for keyword, count in keyword_counter.most_common(top_n):
                result.append({
                    'keyword': keyword,
                    'count': count,
                    'examples': keyword_examples.get(keyword, [])[:3]
                })
            return result
        
        positive_keywords = extract_keywords_with_examples(positive_texts)
        negative_keywords = extract_keywords_with_examples(negative_texts)
        
        return jsonify({
            'code': 200,
            'positive_keywords': positive_keywords,
            'negative_keywords': negative_keywords
        })
        
    except Exception as e:
        import traceback
        print(f"关键词归因错误: {e}")
        traceback.print_exc()
        return jsonify({'error': str(e)}), 500

In [9]:
# ============== 测试功能 ==============
def test_service():
    print("\n" + "="*60)
    print("模型服务本地测试")
    print("="*60)
    # 测试预测功能
    
    test_cases = [
        "这件衣服质量很好，非常满意！",
        "东西太差了，用了一次就坏了",
        "性价比一般，没有想象中好",
        "发货很快，包装完好",
        "客服态度很差，再也不来了"
    ]
    
    for text in test_cases:
        seg = preprocess_single(text)
        vec = vectorizer.transform([seg])
        pred = model.predict(vec)[0]
        sentiment = '好评' if pred == 1 else '差评'
        print(f"评论: {text}")
        print(f"预测: {sentiment}")
        print(f"分词: {seg}\n")
    
    # 词云功能测试
    print("\n--- 词云功能测试 ---")
    test_positive = [
        "质量很好，面料舒适",
        "发货很快，包装完好",
        "性价比很高",
        "客服态度很好"
    ]
    test_negative = [
        "质量太差，洗一次就坏",
        "发货很慢，等了一周",
        "客服态度恶劣",
        "尺码偏小"
    ]
    
    # 处理正面评论
    positive_words = []
    for text in test_positive:
        words = preprocess_for_wordcloud(text)
        positive_words.extend(words)
        print(f"正面评论 '{text}' 分词结果: {words}")
    
    # 处理负面评论
    negative_words = []
    for text in test_negative:
        words = preprocess_for_wordcloud(text)
        negative_words.extend(words)
        print(f"负面评论 '{text}' 分词结果: {words}")
    
    positive_counter = Counter(positive_words)
    negative_counter = Counter(negative_words)
    
    print("\n正面评论词频统计:")
    for word, count in positive_counter.most_common(5):
        print(f"  {word}: {count}")
    
    print("\n负面评论词频统计:")
    for word, count in negative_counter.most_common(5):
        print(f"  {word}: {count}")

In [10]:
test_service()

Building prefix dict from the default dictionary ...



模型服务本地测试


Dumping model to file cache C:\Users\ljh01307\AppData\Local\Temp\jieba.cache
Loading model cost 0.524 seconds.
Prefix dict has been built successfully.


评论: 这件衣服质量很好，非常满意！
预测: 差评
分词: 这件 衣服 质量 满意

评论: 东西太差了，用了一次就坏了
预测: 差评
分词: 东西 太差 一次

评论: 性价比一般，没有想象中好
预测: 差评
分词: 性价比 一般 想象 中好

评论: 发货很快，包装完好
预测: 好评
分词: 发货 很快 包装 完好

评论: 客服态度很差，再也不来了
预测: 差评
分词: 客服 态度 很差 再也 不来


--- 词云功能测试 ---
正面评论 '质量很好，面料舒适' 分词结果: ['质量', '面料', '舒适']
正面评论 '发货很快，包装完好' 分词结果: ['发货', '很快', '包装', '完好']
正面评论 '性价比很高' 分词结果: ['性价比']
正面评论 '客服态度很好' 分词结果: ['客服', '态度']
负面评论 '质量太差，洗一次就坏' 分词结果: ['质量', '一次']
负面评论 '发货很慢，等了一周' 分词结果: ['发货', '一周']
负面评论 '客服态度恶劣' 分词结果: ['客服', '态度恶劣']
负面评论 '尺码偏小' 分词结果: ['尺码', '偏小']

正面评论词频统计:
  质量: 1
  面料: 1
  舒适: 1
  发货: 1
  很快: 1

负面评论词频统计:
  质量: 1
  一次: 1
  发货: 1
  一周: 1
  客服: 1


In [ ]:
# ============== 启动服务 ==============
if __name__ == '__main__':
    # 如果命令行参数包含'test'，执行测试
    if len(sys.argv) > 1 and sys.argv[1] == 'test':
        test_service()
    else:
        print("\n" + "="*60)
        print("启动情感分析模型服务")
        print("="*60)
        print(f"接口地址: http://127.0.0.1:5000")
        print(f"健康检查: http://127.0.0.1:5000/health")
        print(f"单条预测: POST http://127.0.0.1:5000/predict")
        print(f"批量预测: POST http://127.0.0.1:5000/batch_predict")
        print(f"词云生成: POST http://127.0.0.1:5000/wordcloud")
        print(f"词云测试: GET http://127.0.0.1:5000/wordcloud/test")
        print("="*60 + "\n")
        
        # 启动服务
        app.run(host='0.0.0.0', port=5000, debug=False)


启动情感分析模型服务
接口地址: http://127.0.0.1:5000
健康检查: http://127.0.0.1:5000/health
单条预测: POST http://127.0.0.1:5000/predict
批量预测: POST http://127.0.0.1:5000/batch_predict
词云生成: POST http://127.0.0.1:5000/wordcloud
词云测试: GET http://127.0.0.1:5000/wordcloud/test

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://10.136.4.135:5000
Press CTRL+C to quit
127.0.0.1 - - [07/Mar/2026 15:00:25] "POST /wordcloud HTTP/1.1" 200 -
127.0.0.1 - - [07/Mar/2026 15:00:25] "POST /keyword_attribution HTTP/1.1" 200 -
127.0.0.1 - - [07/Mar/2026 15:01:02] "POST /keyword_attribution HTTP/1.1" 200 -
127.0.0.1 - - [07/Mar/2026 15:01:03] "POST /wordcloud HTTP/1.1" 200 -
127.0.0.1 - - [07/Mar/2026 15:01:03] "POST /wordcloud HTTP/1.1" 200 -
127.0.0.1 - - [07/Mar/2026 15:01:03] "POST /keyword_attribution HTTP/1.1" 200 -
127.0.0.1 - - [07/Mar/2026 15:02:27] "POST /predict HTTP/1.1" 200 -
127.0.0.1 - - [07/Mar/2026 15:02:28] "POST /predict HTTP/1.1" 200 -
127.0.0.1 - - [07/Mar/2026 15:02:28] "POST /predict HTTP/1.1" 200 -
127.0.0.1 - - [07/Mar/2026 15:02:29] "POST /predict HTTP/1.1" 200 -
127.0.0.1 - - [07/Mar/2026 15:02:29] "POST /predict HTTP/1.1" 200 -
127.0.0.1 - - [07/Mar/2026 15:02:30] "POST /predict HTTP/1.1" 200 -
127.0.0.